# Lab 6 — Unsupervised Learning: K-Means, PCA & Gaussian Mixtures
**Machine Learning for the Natural Sciences**

Everything so far has been **supervised** — we had labels telling the
model the right answer. This week we remove the labels and ask:
*Can the algorithm find structure on its own?*

Three complementary tools:
- **K-Means** — hard clustering (each sample belongs to exactly one group)
- **Gaussian Mixture Models (GMM)** — soft/probabilistic clustering
- **PCA** — dimensionality reduction (compress many features into fewer)

Together they answer: "What groups exist in my data?" and "Which features
are most informative?"

---

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

RANDOM_STATE = 42

## 1. Prepare Data
For unsupervised learning, we **do not** give the model the species labels.
We'll keep them aside to evaluate how well clustering recovers real groups.

In [ ]:
df = sns.load_dataset("penguins").dropna()

feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df[feature_cols].values
y_true = df["species"].values  # kept aside for evaluation only

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape: {X_scaled.shape}")
print(f"True species: {np.unique(y_true)} (the model won't see these)")

## 2. K-Means Clustering
K-Means partitions data into k groups by minimizing the distance from
each point to its cluster center (centroid).

In [ ]:
# We know there are 3 species — but pretend we don't.
# The elbow method helps us choose k.

inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, "o-", linewidth=2)
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster sum of squares)")
axes[0].set_title("Elbow Method")
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, "s-", linewidth=2, color="green")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k_sil = K_range[np.argmax(silhouettes)]
print(f"Best k by silhouette: {best_k_sil}")

### 🔍 Your Turn
**TODO:** The elbow method looks for a "bend" in the inertia curve.
The silhouette score measures how well-separated clusters are (higher = better).
Do both methods suggest the same k? Is it 3 (the true number of species)?

*Your answer:*

In [ ]:
# Run K-Means with k=3 and compare to true species
km3 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
km_labels = km3.fit_predict(X_scaled)

# Cross-tabulation: how do clusters map to species?
ct = pd.crosstab(y_true, km_labels, rownames=["True Species"],
                  colnames=["K-Means Cluster"])
print("K-Means Clusters vs. True Species:")
print(ct)
print(f"\nAdjusted Rand Index: {adjusted_rand_score(y_true, km_labels):.3f}")
print("(1.0 = perfect match, 0.0 = random)")

## 3. PCA: Dimensionality Reduction
PCA finds the axes of maximum variance in your data. It's useful for
visualization (compress 4+ features into 2 for plotting) and for
understanding which directions in feature space carry the most information.

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# How much variance does each component explain?
print("Explained Variance Ratio:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)")
print(f"\nPC1 + PC2 = {pca.explained_variance_ratio_[:2].sum():.1%} of total variance")

In [ ]:
# Cumulative variance plot
plt.figure(figsize=(6, 4))
cumulative = np.cumsum(pca.explained_variance_ratio_)
plt.bar(range(1, len(cumulative)+1), pca.explained_variance_ratio_,
        alpha=0.6, label="Individual")
plt.step(range(1, len(cumulative)+1), cumulative, where="mid",
         color="red", linewidth=2, label="Cumulative")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Scree Plot")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA biplot — visualize data AND feature loadings together
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter the data in PC space, colored by true species
for species in np.unique(y_true):
    mask = y_true == species
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6, label=species, s=30)

# Overlay the loadings as arrows
loadings = pca.components_[:2].T
scale = 3  # scale arrows for visibility
for i, feat in enumerate(feature_cols):
    ax.arrow(0, 0, loadings[i, 0]*scale, loadings[i, 1]*scale,
             head_width=0.1, head_length=0.05, fc="black", ec="black")
    ax.text(loadings[i, 0]*scale*1.15, loadings[i, 1]*scale*1.15,
            feat.replace("_mm", "").replace("_g", ""),
            fontsize=9, ha="center")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("PCA Biplot — Penguins")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🔍 Your Turn
**TODO:** Look at the biplot arrows. Which original features align most
with PC1? Which with PC2? What does this tell you about what each
principal component "means" in penguin biology?

*Your answer:*

### PCA + K-Means: Visualize Clusters in PC Space

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# True species
for species in np.unique(y_true):
    mask = y_true == species
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6, label=species, s=30)
axes[0].set_title("True Species")
axes[0].legend()

# K-Means clusters
for c in range(3):
    mask = km_labels == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.6, label=f"Cluster {c}", s=30)
axes[1].set_title("K-Means Clusters")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Gaussian Mixture Models
K-Means assigns each point to exactly one cluster (hard assignment).
GMM gives **probabilities** — a point might be 80% Cluster A, 20% Cluster B.
This is more realistic for natural data where groups overlap.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(X_scaled)
gmm_probs = gmm.predict_proba(X_scaled)

print("GMM Clusters vs. True Species:")
print(pd.crosstab(y_true, gmm_labels, rownames=["True"], colnames=["GMM"]))
print(f"\nAdjusted Rand Index: {adjusted_rand_score(y_true, gmm_labels):.3f}")

In [ ]:
# Show some uncertain assignments
proba_df = pd.DataFrame(gmm_probs, columns=[f"P(Cluster {i})" for i in range(3)])
proba_df["Max Probability"] = proba_df.max(axis=1)
proba_df["True Species"] = y_true

uncertain = proba_df[proba_df["Max Probability"] < 0.80].sort_values("Max Probability")
print(f"\nSamples with < 80% confidence: {len(uncertain)}")
if len(uncertain) > 0:
    print(uncertain.head(10).to_string(index=False))

### BIC for choosing the number of components

In [ ]:
bics = []
for n in range(1, 8):
    gm = GaussianMixture(n_components=n, random_state=RANDOM_STATE)
    gm.fit(X_scaled)
    bics.append(gm.bic(X_scaled))

plt.figure(figsize=(6, 4))
plt.plot(range(1, 8), bics, "o-", linewidth=2)
plt.xlabel("Number of Components")
plt.ylabel("BIC (lower is better)")
plt.title("GMM: Bayesian Information Criterion")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_n_bic = np.argmin(bics) + 1
print(f"Best number of components by BIC: {best_n_bic}")

### 🔍 Your Turn
**TODO:**
1. Compare the K-Means and GMM cross-tabulations. Which did a better
   job recovering the true species?

   *Your answer:*

2. In what kinds of natural science problems would soft (probabilistic)
   clustering be more useful than hard clustering? Give an example from
   your Data Path.

   *Your answer:*

3. The BIC plot helps choose number of components like the elbow method
   helps choose k. Do they agree?

   *Your answer:*

## 5. Using PCA Components as Features
PCA isn't just for visualization — you can use the PC axes as features
in a supervised model. This is especially useful when you have many
correlated features.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y_true)

# Compare: all 4 features vs. 2 PCA components
rf_full = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_scaled, y_enc, cv=5
)
rf_pca2 = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_pca[:, :2], y_enc, cv=5
)

print(f"RF on all 4 features:  {rf_full.mean():.3f} +/- {rf_full.std():.3f}")
print(f"RF on 2 PCA components: {rf_pca2.mean():.3f} +/- {rf_pca2.std():.3f}")

---
## What to Submit
1. PDF with answers to all **🔍 Your Turn** sections
2. At least two figures: the PCA biplot and a K-Means/GMM comparison
3. Cross-tabulation of clusters vs. true species

## Also Due This Week
**Data Adventure 3** — apply PCA, clustering, metrics, and imbalance
handling to your Data Path dataset. See the Data Adventure 3 prompt.

## What's Next
**Week 8** introduces **Neural Networks** — a completely different paradigm
where the model learns its own features through layers of transformations.